# HS4002 Week 11

## Panel Data Models

Panel data follows the **same units** (people, players, countries) across multiple time points, allowing us to control for unobserved unit-level heterogeneity.

In Python, the `linearmodels` library provides panel regression — equivalent to R's `plm` package.

We'll use NBA player data (2013–2017) to ask: **how does a player's age affect their scoring performance?**

In [1]:
# !pip install pandas numpy plotnine linearmodels stargazer

import pandas as pd
import numpy as np
from plotnine import *
import warnings
warnings.filterwarnings('ignore')
from linearmodels.panel import PanelOLS, RandomEffects, PooledOLS

## Setup: Load and Prepare the NBA Dataset

In [2]:
# Data source: https://www.kaggle.com/datasets/drgilermo/nba-players-stats
# NBA player statistics 1950-2017 from basketball-reference.com
raw_df = pd.read_csv("bball.csv").iloc[:, 1:]  # drop the unnamed index column

df = (
	raw_df[raw_df["Year"] >= 2013]
	.groupby(["Player", "Year"])
	.agg(
		games_played    = ("G",    "sum"),
		games_started   = ("GS",   "sum"),
		minutes_played  = ("MP",   "sum"),
		age             = ("Age",  "mean"),
		personal_fouls  = ("PF",   "sum"),
		points          = ("PTS",  "sum"),
		true_shooting   = ("TS%",  "mean"),
		assists         = ("AST",  "sum"),
		steals          = ("STL",  "sum"),
		blocks          = ("BLK",  "sum"),
		fga             = ("FGA",  "sum"),
		fg              = ("FG",   "sum"),
		threeP          = ("3P",   "sum"),
		threePA         = ("3PA",  "sum"),
		twoP            = ("2P",   "sum"),
		twoPA           = ("2PA",  "sum"),
		ft              = ("FT",   "sum"),
		fta             = ("FTA",  "sum"),
		def_rb          = ("DRB",  "sum"),
		off_rb          = ("ORB",  "sum"),
	)
	.assign(ppg=lambda x: x["points"] / x["games_played"])
	.reset_index()
)
print(df.shape)
df.head()

(2403, 23)


,Player,Year,games_played,games_started,minutes_played,age,personal_fouls,points,true_shooting,assists,...,fg,threeP,threePA,twoP,twoPA,ft,fta,def_rb,off_rb,ppg
0,A.J. Hammons,2017.0,22.0,0.0,163.0,24.0,21.0,48.0,0.472000,4.0,...,17.0,5.0,10.0,12.0,32.0,9.0,20.0,28.0,8.0,2.181818
1,A.J. Price,2013.0,57.0,22.0,1278.0,26.0,73.0,441.0,0.501000,205.0,...,161.0,70.0,200.0,91.0,213.0,49.0,62.0,94.0,20.0,7.736842
2,A.J. Price,2014.0,28.0,0.0,99.0,27.0,5.0,44.0,0.469000,13.0,...,19.0,6.0,22.0,13.0,24.0,0.0,2.0,9.0,1.0,1.571429
3,A.J. Price,2015.0,52.0,0.0,648.0,28.0,30.0,266.0,0.376750,92.0,...,102.0,30.0,114.0,72.0,160.0,32.0,48.0,52.0,12.0,5.115385
4,Aaron Brooks,2013.0,106.0,40.0,1994.0,28.0,188.0,752.0,0.500667,228.0,...,286.0,100.0,268.0,186.0,364.0,80.0,104.0,134.0,26.0,7.094340


## Setting up the Panel Index

`linearmodels` requires a **MultiIndex** of (entity, time). Equivalent to the `index=c('Player','Year')` argument in R's `plm()`.

In [3]:
# Set the panel index — required by linearmodels
df_panel = df.set_index(['Player', 'Year'])

# A Tutorial for Panel Models

## Pooled OLS

Equivalent to a regular `lm()` in R — ignores the panel structure.

In [4]:
pool_model = PooledOLS.from_formula('ppg ~ age + I(age**2)', data=df_panel)
pool_result = pool_model.fit()
print(pool_result.summary)

                          PooledOLS Estimation Summary                          
Dep. Variable:                    ppg   R-squared:                        0.6757
Estimator:                  PooledOLS   R-squared (Between):              0.6103
No. Observations:                2403   R-squared (Within):               0.0318
Date:                Sat, Oct 31 2026   R-squared (Overall):              0.6757
Time:                        15:26:35   Log-likelihood                   -7587.5
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      2501.2
Entities:                         817   P-value                           0.0000
Avg Obs:                       2.9412   Distribution:                  F(2,2401)
Min Obs:                       1.0000                                           
Max Obs:                       5.0000   F-statistic (robust):             2501.2
                            

## Fixed Effects (Within) Model

Adds `EntityEffects` to control for time-invariant player characteristics. Equivalent to `plm(..., model='within')` in R.

In [5]:
fe_model = PanelOLS.from_formula('ppg ~ age + I(age**2) + EntityEffects', data=df_panel)
fe_result = fe_model.fit()
print(fe_result.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:                    ppg   R-squared:                        0.1774
Estimator:                   PanelOLS   R-squared (Between):             -66.812
No. Observations:                2403   R-squared (Within):               0.1774
Date:                Sat, Oct 31 2026   R-squared (Overall):             -43.473
Time:                        15:26:35   Log-likelihood                   -5167.4
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      170.80
Entities:                         817   P-value                           0.0000
Avg Obs:                       2.9412   Distribution:                  F(2,1584)
Min Obs:                       1.0000                                           
Max Obs:                       5.0000   F-statistic (robust):             170.80
                            

Interpret the coefficients. How does the fixed-effects estimate differ from the pooled OLS estimate?

The fixed-effects model shows an inverted U-shaped relationship between age and points per game (PPG). The coefficient for age is positive (5.7447), while the coefficient for age squared is negative (-0.1078). P<0.001, both are statistically significant. 

## Random Effects Model

Equivalent to `plm(..., model='random')` in R.

In [6]:
re_model = RandomEffects.from_formula('ppg ~ age + I(age**2)', data=df_panel)
re_result = re_model.fit()
print(re_result.summary)

                        RandomEffects Estimation Summary                        
Dep. Variable:                    ppg   R-squared:                        0.3804
Estimator:              RandomEffects   R-squared (Between):              0.6371
No. Observations:                2403   R-squared (Within):               0.0412
Date:                Sat, Oct 31 2026   R-squared (Overall):              0.6573
Time:                        15:26:35   Log-likelihood                   -5810.5
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      737.02
Entities:                         817   P-value                           0.0000
Avg Obs:                       2.9412   Distribution:                  F(2,2401)
Min Obs:                       1.0000                                           
Max Obs:                       5.0000   F-statistic (robust):             737.02
                            

## Model Diagnostics

### Lagrange Multiplier Test

Tests whether individual effects are present (i.e., whether FE/RE is needed over pooled OLS). Equivalent to R's `plmtest(fe_model, effect='individual')`.

In [7]:
# Breusch-Pagan LM test for random effects
# Under H0 (no individual effects), we prefer pooled OLS.
# Equivalent to R's plmtest(..., effect='individual', type='bp').
from scipy.stats import chi2

resid = pool_result.resids
e_i   = resid.groupby(level=0).sum()
ssr   = float((resid ** 2).sum())
A1    = float((e_i ** 2).sum()) / ssr - 1
T_i   = resid.groupby(level=0).size()
n_obs = resid.shape[0]
lm_stat = (n_obs ** 2) / (2 * (float((T_i ** 2).sum()) - n_obs)) * A1 ** 2
lm_pval = chi2.sf(lm_stat, df=1)

print('LM test for individual effects:')
print(f'LM statistic = {lm_stat:.4f}')
print('df = 1')
print(f'p-value = {lm_pval:.4f}')

LM test for individual effects:
LM statistic = 2381.2182
df = 1
p-value = 0.0000


### Hausman Test

Tests whether RE estimates are consistent (i.e., whether FE or RE is preferred). Under H0, RE is consistent and more efficient. Equivalent to R's `phtest(fe_model, re_model)`.

In [8]:
# Hausman test: compare FE and RE estimates
# Under H0 (RE consistent), the difference in coefficients should be small
from scipy.stats import chi2

common_params = fe_result.params.index.intersection(re_result.params.index)

beta_diff = fe_result.params[common_params] - re_result.params[common_params]
cov_diff  = (
	fe_result.cov.loc[common_params, common_params] -
	re_result.cov.loc[common_params, common_params]
)

h_stat = float(beta_diff.T @ np.linalg.inv(cov_diff.values) @ beta_diff)
p_val  = chi2.sf(h_stat, df=len(common_params))
print(f'Hausman statistic = {h_stat:.4f}')
print(f'df = {len(common_params)}')
print(f'p-value = {p_val:.4f}')
print('Prefer FE over RE:', p_val < 0.05)

Hausman statistic = 285.0094
df = 2
p-value = 0.0000
Prefer FE over RE: True


## Comparing the Models

As in earlier weeks, `stargazer` collects the fitted models into one regression table. Reading pooled OLS, fixed effects, and random effects side by side is easier than scrolling through three summaries. Equivalent to R's `stargazer(lm1, fe_glm1, re_glm1)`.

In [9]:
from stargazer.stargazer import Stargazer

sg = Stargazer([pool_result, fe_result, re_result])
sg.title('Points per Game and Age')

# Label the models and variables readably, rather than by raw column name
sg.dependent_variable_name('Points per game')
sg.custom_columns(['Pooled OLS', 'Fixed Effects', 'Random Effects'], [1, 1, 1])
sg.covariate_order(['age', 'I(age ** 2)'])
sg.rename_covariates({
	'age':         'Age',
	'I(age ** 2)': 'Age-squared',
})

# Ending the cell on the object itself renders the table in the notebook
sg

# Independent Analysis: Has Sports Science Extended Career Peaks?

Compare players from 2008–2012 with players from 2013–2017. Does age affect scoring differently across the two eras?

In [10]:
# (1) create two data sets, one for each era.

In [24]:
# Load original dataset
df_all = pd.read_csv("bball.csv")

# Keep 2008–2017
df_all = df_all[df_all["Year"].between(2008, 2017)].copy()

# Prefer total-season records for traded players
df_all["is_tot"] = df_all["Tm"].eq("TOT")

df_all = (
    df_all.sort_values("is_tot", ascending=False)
    .drop_duplicates(subset=["Player", "Year"], keep="first")
    .copy()
)

# Calculate PPG
df_all["ppg"] = df_all["PTS"] / df_all["G"]

# Rename Age
df_all = df_all.rename(columns={"Age": "age"})

# Keep relevant variables
df_all = df_all[["Player", "Year", "age", "ppg"]]

# Remove missing / invalid values
df_all = df_all.replace([np.inf, -np.inf], np.nan).dropna()

# Create two datasets
df_early = df_all[df_all["Year"].between(2008, 2012)].copy()
df_late = df_all[df_all["Year"].between(2013, 2017)].copy()

# Set panel index
early_panel = df_early.set_index(["Player", "Year"])
late_panel = df_late.set_index(["Player", "Year"])

# Check duplicates
print("Early duplicates:", early_panel.index.duplicated().sum())
print("Late duplicates:", late_panel.index.duplicated().sum())

print("Early shape:", early_panel.shape)
print("Late shape:", late_panel.shape)

Early duplicates: 0
Late duplicates: 0
Early shape: (2266, 2)
Late shape: (2403, 2)


In [11]:
# (2) fit models for each of the two data sets

In [25]:
pool_early, fe_early, re_early = fit_models(early_panel)

pool_late, fe_late, re_late = fit_models(late_panel)

In [12]:
# (3) run diagnostics, compare, and interpret for 2008-2012.

In [26]:
from linearmodels.panel import compare

print(compare({
    "Pooled OLS": pool_early,
    "Fixed Effects": fe_early,
    "Random Effects": re_early
}))

print("\nF-test for individual effects:")
print(fe_early.f_pooled)

                            Model Comparison                           
                            Pooled OLS  Fixed Effects    Random Effects
-----------------------------------------------------------------------
Dep. Variable                      ppg            ppg               ppg
Estimator                    PooledOLS       PanelOLS     RandomEffects
No. Observations                  2266           2266              2266
Cov. Est.                   Unadjusted     Unadjusted        Unadjusted
R-squared                       0.0157         0.2144            0.0413
R-Squared (Within)              0.0945         0.2144            0.1842
R-Squared (Between)            -0.0759        -0.4118           -0.1062
R-Squared (Overall)             0.0157        -0.1696           -0.0664
F-statistic                     18.027         207.55            131.86
P-value (F-stat)                0.0000         0.0000            0.0000
=====================     ============   ============   ========

For Pooled OLS: ppg = 1.8654age - 0.0347age2 - 16.183 + r
For Fixed effects: ppg = 5.1097age -0.1024age2 -53.388 + r
For random effects: ppg = 3.8451age -0.0743age2 -41.296 + r
All shows an inverted U-shaped relationship between age and scoring performance, as coefficient of age2 all negative. This means that players' scoring performance initially increases with age and then declines. 
Fixed Effects model has the highest within R-squared, which means that it can explains 21.44% changes in scoring performance within individual players over time. 
The F-test for individual effects is statistically significant (F = 15.0041, p < 0.001), indicating that player-specific fixed effects are important.
Fixed Effects model has an F-statistic of 207.55 (p < 0.001), indicating that age and age squared are jointly statistically significant in explaining within-player variations in PPG.

In [13]:
# (4) run diagnostics, compare, and interpret for 2013-2017.

In [27]:
print(compare({
    "Pooled OLS": pool_late,
    "Fixed Effects": fe_late,
    "Random Effects": re_late
}))

print("\nF-test for individual effects:")
print(fe_late.f_pooled)

                            Model Comparison                           
                            Pooled OLS  Fixed Effects    Random Effects
-----------------------------------------------------------------------
Dep. Variable                      ppg            ppg               ppg
Estimator                    PooledOLS       PanelOLS     RandomEffects
No. Observations                  2403           2403              2403
Cov. Est.                   Unadjusted     Unadjusted        Unadjusted
R-squared                       0.0171         0.1776           -0.0026
R-Squared (Within)              0.0856         0.1776            0.1507
R-Squared (Between)            -0.0953        -0.3525           -0.0832
R-Squared (Overall)             0.0171        -0.1132           -0.0527
F-statistic                     20.918         170.98            107.42
P-value (F-stat)                0.0000         0.0000            0.0000
=====================     ============   ============   ========

For Pooled OLS: ppg = 1.8688age - 0.0331age2 - 17.4843 + r
For Fixed effects: ppg = 5.7454age -0.1078age2 -66.358 + r
For random effects: ppg = 3.8623age -0.0703age2 -44.681 + r
All shows an inverted U-shaped relationship between age and scoring performance, as coefficient of age2 all negative. This means that players' scoring performance initially increases with age and then declines. 
Fixed Effects model has the highest within R-squared, which means that it can explains 17.76% changes in scoring performance within individual players over time. 
The F-test for individual effects is statistically significant (F = 12.4940, p < 0.001), indicating that player-specific fixed effects are important.
Fixed Effects model has an F-statistic of 170.98 (p < 0.001), indicating that age and age squared are jointly statistically significant in explaining within-player variations in PPG.

In [14]:
# (5) what differences do you find in the age-scoring relationship across the two eras?

In [28]:
def peak_age(model):
    b1 = model.params["age"]
    b2 = model.params["I(age ** 2)"]
    return -b1 / (2 * b2)

results = pd.DataFrame({
    "Model": ["Pooled OLS", "Fixed Effects", "Random Effects"],
    "2008–2012": [
        peak_age(pool_early),
        peak_age(fe_early),
        peak_age(re_early)
    ],
    "2013–2017": [
        peak_age(pool_late),
        peak_age(fe_late),
        peak_age(re_late)
    ]
})

results["Difference"] = results["2013–2017"] - results["2008–2012"]

print(results.round(2))

            Model  2008–2012  2013–2017  Difference
0      Pooled OLS      26.90      28.23        1.33
1   Fixed Effects      24.96      26.64        1.68
2  Random Effects      25.88      27.45        1.57


The estimated peak age in all three models was delayed by approximately 1.3–1.7 years. This means that NBA players reached their peak scoring performance at a later age in 2013–2017 compared to 2008–2012. This may be consistent with improvements in sports science.